# 11 — Feature Engineering and Preprocessing

## 1. Objective and scope boundary

This notebook freezes the feature policy only. It decides which features may proceed into feature-engineering experiments, without creating those features or performing preprocessing.

```text
Notebook 10: What does the data tell us?
        ↓
Notebook 11 policy freeze: Which features are allowed to proceed?
        ↓
Next work: How are approved features created?
```

> The policy freeze establishes feature eligibility before transformation. This prevents preprocessing code from implicitly deciding which columns are allowed into the model.

An approved candidate is allowed into deterministic feature-creation design; it is **not** guaranteed to be a final model feature.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from urban_ops.features.policy import (
    PolicyStatus,
    feature_policy_table,
    load_feature_policy,
    validate_feature_policy_evidence,
)

POLICY_PATH = PROJECT_ROOT / 'configs/features/resolution_risk_baseline.yaml'

def file_state(path):
    return (sha256(path.read_bytes()).hexdigest(), path.stat().st_mtime_ns)

latest_pointer = PROJECT_ROOT / 'data/splits/resolution_risk/latest.json'
latest_payload = json.loads(latest_pointer.read_text(encoding='utf-8'))
configured_run = Path(latest_payload['run_path'])
split_run = configured_run if configured_run.is_absolute() else PROJECT_ROOT / configured_run
governed_inputs = [
    POLICY_PATH,
    latest_pointer,
    split_run / 'train.parquet',
    split_run / 'validation.parquet',
    split_run / 'test.parquet',
    split_run / 'split_metadata.json',
    split_run / 'split_rules_snapshot.yaml',
]
governed_inputs.extend(sorted((PROJECT_ROOT / 'reports/11_split_aware_eda').rglob('*')))
governed_inputs = tuple(path for path in governed_inputs if path.is_file())
before_states = {str(path.relative_to(PROJECT_ROOT)): file_state(path) for path in governed_inputs}

pd.Series({'governed_input_count': len(before_states), 'split_id': latest_payload['split_id']})

governed_input_count                                   55
split_id                20260806T135114Z_9d945cb2da0eecfc
dtype: object

## 2. Notebook 10 handoff authority

Notebook 10 completed its analysis and reconciliation, but reported `model_ready: False`. That is correct: EDA describes the data and recommends possibilities, but it does not freeze which representations are permitted to proceed.

The authority order is: Step 4 leakage and prediction-time availability; Notebook 10 leakage audit; Notebook 10's newer `eda_status`; train-only structure evidence; then the older `baseline_decision`. A statistically interesting feature can remain excluded or conditional because governance takes priority over descriptive EDA patterns.

In [2]:
policy = load_feature_policy(POLICY_PATH)
policy_table = feature_policy_table(policy)
display(pd.Series(policy.notebook_10_handoff, name='Notebook 10 handoff'))
display(pd.DataFrame(policy.authority_hierarchy).sort_values('priority'))

split_id            20260806T135114Z_9d945cb2da0eecfc
integrity                                        PASS
reconciliation                                   PASS
step_9a_decision                             COMPLETE
model_ready                                     False
Name: Notebook 10 handoff, dtype: object

,priority,authority,source
0,1,Step 4 leakage and prediction-time availability,docs/leakage_policy.md
1,2,Notebook 10 leakage audit,reports/11_split_aware_eda/tables/leakage_audi...
2,3,Notebook 10 EDA statuses,notebooks/10_split_aware_eda.ipynb
3,4,"Notebook 10 missingness, cardinality, and outl...",reports/11_split_aware_eda/
4,5,Notebook 10 baseline recommendations,reports/11_split_aware_eda/tables/baseline_fea...


## 3. Feature-policy decision rules

A feature receives permission to proceed only after all of these gates pass:

1. available at the complaint-creation prediction moment;
2. safe under Step 4 leakage governance;
3. non-null with usable train variation;
4. the approved representation rather than an alternative or redundant form; and
5. no unresolved policy or temporal-generalization blocker.

Target-rate differences alone can never grant feature-creation eligibility.

## 4. Approved first-pass candidates

The four preferred temporal representations are frozen as candidates. Their common source is retained, but no calendar field is derived during this work.

In [3]:
approved = policy_table.loc[policy_table['policy_status'].eq('APPROVED_CANDIDATE')]
display(approved[['feature_name', 'source_column', 'policy_status', 'eda_status', 'reason', 'phase_2_allowed']])
assert tuple(approved['feature_name']) == policy.feature_creation_allow_list

,feature_name,source_column,policy_status,eda_status,reason,phase_2_allowed
1,created_hour,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred creation-time-safe representation fo...,True
2,created_day_of_week,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred machine-friendly creation-time weekd...,True
3,created_month,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred interpretable creation-time seasonal...,True
4,is_weekend,created_date,APPROVED_CANDIDATE,CANDIDATE,Simple creation-time-safe operational grouping...,True


## 5. Alternative and redundant representations

Names and numeric calendar codes carry the same essential information for day and month, so the first baseline selects one representation. Quarter and week-of-year overlap with the preferred month representation and remain under redundancy review.

In [4]:
representation_review = policy_table.loc[policy_table['policy_status'].isin([
    'ALTERNATIVE_REPRESENTATION', 'REVIEW_REDUNDANCY', 'REVIEW'
])]
display(representation_review[['feature_name', 'policy_status', 'preferred_counterpart', 'redundancy_status', 'reason', 'phase_2_allowed']])

,feature_name,policy_status,preferred_counterpart,redundancy_status,reason,phase_2_allowed
5,created_day_name,ALTERNATIVE_REPRESENTATION,created_day_of_week,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
6,created_month_name,ALTERNATIVE_REPRESENTATION,created_month,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
7,created_quarter,REVIEW_REDUNDANCY,created_month,OVERLAPS_PREFERRED,Coarser seasonal representation that overlaps ...,False
8,created_week_of_year,REVIEW_REDUNDANCY,created_month,OVERLAPS_PREFERRED,Seasonal representation that overlaps strongly...,False
9,created_day_of_month,REVIEW,NaN,NONE,"Creation-time safe, but current EDA does not j...",False


## 6. Conditional features

`created_year` may encode time progression or regime rather than a durable operational pattern. Geography remains unresolved because the current authority does not prove creation-time availability and immutability. Missingness strategies and apparent target associations do not resolve that uncertainty.

In [5]:
conditional = policy_table.loc[policy_table['policy_status'].eq('CONDITIONAL')]
display(conditional[['feature_name', 'prediction_time_status', 'leakage_status', 'reason', 'phase_2_allowed']])
assert not conditional['phase_2_allowed'].any()

,feature_name,prediction_time_status,leakage_status,reason,phase_2_allowed
10,created_year,AVAILABLE,SAFE,May encode temporal progression or regime rath...,False
11,borough,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
12,location_type,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
13,incident_zip,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
14,latitude,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
15,longitude,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False


## 7. Excluded features

Exclusions retain explicit reasons: leakage/post-creation/target-derived, identifier, all-null, or zero variance. `unique_key` remains available only for traceability. `due_date` remains blocked because its prediction-time availability and mutability are unproven.

In [6]:
excluded = policy_table.loc[policy_table['policy_status'].str.startswith('EXCLUDE_')]
display(excluded[['feature_name', 'policy_status', 'prediction_time_status', 'leakage_status', 'variation_status', 'reason', 'phase_2_allowed']])
assert not excluded['phase_2_allowed'].any()

,feature_name,policy_status,prediction_time_status,leakage_status,variation_status,reason,phase_2_allowed
16,unique_key,EXCLUDE_IDENTIFIER,AVAILABLE,BLOCKED,HAS_VARIATION,Identifier is retained for traceability only a...,False
17,descriptor_2,EXCLUDE_ALL_NULL,UNRESOLVED,CONDITIONAL,ALL_NULL,Training evidence identifies the field as enti...,False
18,agency,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
19,agency_name,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
20,complaint_type,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
21,descriptor,EXCLUDE_ZERO_VARIANCE,UNRESOLVED,CONDITIONAL,ZERO_VARIANCE,Training evidence shows no useful variation; e...,False
22,open_data_channel_type,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Training evidence shows one fixed channel valu...,False
23,closed_date,EXCLUDE_LEAKAGE,UNAVAILABLE,BLOCKED,HAS_VARIATION,Closure outcome is unavailable at complaint cr...,False
24,due_date,EXCLUDE_LEAKAGE,UNRESOLVED,BLOCKED,HAS_VARIATION,Target input whose creation-time timing and mu...,False
25,status,EXCLUDE_LEAKAGE,UNAVAILABLE,BLOCKED,ZERO_VARIANCE,Final mutable outcome status is post-creation ...,False


## 8. Frozen feature policy

This is the complete required policy table. The YAML remains the machine-readable source of truth; this view is generated from the validated loader.

In [7]:
required_columns = [
    'feature_name', 'source_column', 'policy_status', 'prediction_time_status',
    'leakage_status', 'eda_status', 'redundancy_status', 'reason', 'phase_2_allowed'
]
display(policy_table[required_columns])
assert policy_table['feature_name'].is_unique
assert policy_table[required_columns].notna().all().all()

,feature_name,source_column,policy_status,prediction_time_status,leakage_status,eda_status,redundancy_status,reason,phase_2_allowed
0,created_date,created_date,SOURCE_ONLY,AVAILABLE,SAFE,REVIEW,SOURCE_FOR_DERIVATIONS,Required to derive temporal features; the raw ...,False
1,created_hour,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred creation-time-safe representation fo...,True
2,created_day_of_week,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred machine-friendly creation-time weekd...,True
3,created_month,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred interpretable creation-time seasonal...,True
4,is_weekend,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Simple creation-time-safe operational grouping...,True
5,created_day_name,created_date,ALTERNATIVE_REPRESENTATION,AVAILABLE,SAFE,ALTERNATIVE_REPRESENTATION,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
6,created_month_name,created_date,ALTERNATIVE_REPRESENTATION,AVAILABLE,SAFE,ALTERNATIVE_REPRESENTATION,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
7,created_quarter,created_date,REVIEW_REDUNDANCY,AVAILABLE,SAFE,REVIEW_REDUNDANCY,OVERLAPS_PREFERRED,Coarser seasonal representation that overlaps ...,False
8,created_week_of_year,created_date,REVIEW_REDUNDANCY,AVAILABLE,SAFE,REVIEW_REDUNDANCY,OVERLAPS_PREFERRED,Seasonal representation that overlaps strongly...,False
9,created_day_of_month,created_date,REVIEW,AVAILABLE,SAFE,REVIEW,NONE,"Creation-time safe, but current EDA does not j...",False


## 9. Policy validation

The validator reconciles the policy against Notebook 10's complete baseline inventory, Step 4/Notebook 10 leakage audit, all-null evidence, and zero-variance evidence. The final check also confirms every governed split artifact and Notebook 10 report has the same hash and modification time as at notebook start.

In [8]:
validate_feature_policy_evidence(policy)
after_states = {str(path.relative_to(PROJECT_ROOT)): file_state(path) for path in governed_inputs}
boundary_flags = [
    'transformations_implemented', 'missing_values_imputed',
    'rare_categories_grouped', 'encoder_fitted', 'scaler_fitted',
    'column_transformer_built', 'feature_matrix_created', 'model_trained'
]
policy_checks = pd.Series({
    'notebook_10_complete': policy.notebook_10_handoff['step_9a_decision'] == 'COMPLETE',
    'notebook_10_not_model_ready': policy.notebook_10_handoff['model_ready'] is False,
    'exact_feature_creation_allow_list': policy.feature_creation_allow_list == ('created_hour', 'created_day_of_week', 'created_month', 'is_weekend'),
    'only_approved_candidates_allowed': policy_table.loc[policy_table['phase_2_allowed'], 'policy_status'].eq('APPROVED_CANDIDATE').all(),
    'implementation_scope_respected': all(policy.implementation_boundary[flag] is False for flag in boundary_flags),
    'governed_sources_unchanged': before_states == after_states,
}, name='passed')
display(policy_checks)
assert policy_checks.all()

notebook_10_complete                 True
notebook_10_not_model_ready          True
exact_feature_creation_allow_list    True
only_approved_candidates_allowed     True
implementation_scope_respected       True
governed_sources_unchanged           True
Name: passed, dtype: bool

## 10. Completion decision

The policy freeze is complete when the policy and evidence reconcile, only approved candidates are allowed to proceed, and governed inputs remain unchanged. The frozen decision is now consumed by deterministic feature creation below.

In [9]:
pd.Series({
    'policy_decision': 'COMPLETE' if policy_checks.all() else 'NOT_COMPLETE',
    'model_ready': False,
    'feature_creation_allow_list': ', '.join(policy.feature_creation_allow_list),
    'policy_handoff': 'READY_FOR_DETERMINISTIC_CREATION',
})

policy_decision                                                         COMPLETE
model_ready                                                                False
feature_creation_allow_list    created_hour, created_day_of_week, created_mon...
policy_handoff                                  READY_FOR_DETERMINISTIC_CREATION
dtype: object

## Deterministic Feature Creation

The frozen policy decided **which** derived features are allowed. This section implements **how** those approved features are calculated.

Deterministic creation learns no statistics from training data, so it requires no fitting. The same timestamp always produces the same values, independent of row order, targets, validation/test behavior, randomness, locale, current time, or external services.

Missing-value handling, category grouping, encoding, scaling, outlier transformation, persisted model matrices, and model training remain intentionally deferred.

### A. Frozen policy handoff

The builder consumes the already validated policy. It does not maintain a second allow-list or reinterpret EDA evidence. `created_date` remains a source-only dependency rather than a direct baseline feature.

In [10]:
from urban_ops.eda.pipeline import load_eda_config
from urban_ops.eda.source import load_verified_split, resolve_split_run
from urban_ops.features.temporal import (
    approved_temporal_feature_names,
    build_feature_reconciliation_table,
    build_temporal_validation_table,
    derive_split_temporal_features,
)

EDA_CONFIG_PATH = PROJECT_ROOT / 'configs/eda/resolution_risk.yaml'
eda_config = load_eda_config(EDA_CONFIG_PATH)
split_run_path = resolve_split_run(
    split_root=eda_config.split_root,
    latest_pointer=eda_config.latest_pointer,
)
split_source = load_verified_split(
    run_path=split_run_path,
    latest_pointer=eda_config.latest_pointer,
    required_completion_status=eda_config.required_completion_status,
    identifier_column=eda_config.identifier_column,
    target_column=eda_config.target_column,
    timestamp_column=eda_config.timestamp_column,
)
source_frames = {
    'train': split_source.train,
    'validation': split_source.validation,
    'test': split_source.test,
}
source_snapshots = {name: frame.copy(deep=True) for name, frame in source_frames.items()}
pd.Series({name: len(frame) for name, frame in source_frames.items()}, name='source_rows')

train         23699
validation     6762
test           5499
Name: source_rows, dtype: int64

### B. Approved derivation allow-list

A feature is derived only when the frozen policy marks it `APPROVED_CANDIDATE` and allows it to proceed. Conditional geography and non-preferred calendar representations remain outside the created baseline set.

In [11]:
approved_names = approved_temporal_feature_names(policy)
approved_policy_rows = policy_table.loc[policy_table['feature_name'].isin(approved_names)]
display(approved_policy_rows[['feature_name', 'source_column', 'policy_status', 'phase_2_allowed', 'reason']])
assert approved_names == policy.feature_creation_allow_list
assert policy.by_name['created_date'].policy_status is PolicyStatus.SOURCE_ONLY

,feature_name,source_column,policy_status,phase_2_allowed,reason
1,created_hour,created_date,APPROVED_CANDIDATE,True,Preferred creation-time-safe representation fo...
2,created_day_of_week,created_date,APPROVED_CANDIDATE,True,Preferred machine-friendly creation-time weekd...
3,created_month,created_date,APPROVED_CANDIDATE,True,Preferred interpretable creation-time seasonal...
4,is_weekend,created_date,APPROVED_CANDIDATE,True,Simple creation-time-safe operational grouping...


### C. Source-column contract

Published split timestamps must already be non-null, timezone-aware UTC datetimes. The builder does not parse strings, coerce malformed values, infer a timezone, or use the local machine timezone. Calendar components therefore retain the authoritative UTC interpretation used by cleaning and splitting.

In [12]:
source_contract = pd.DataFrame([
    {
        'split_name': name,
        'row_count': len(frame),
        'created_date_dtype': str(frame['created_date'].dtype),
        'timezone': str(frame['created_date'].dt.tz),
        'null_count': int(frame['created_date'].isna().sum()),
        'contract_valid': str(frame['created_date'].dt.tz) == 'UTC' and not frame['created_date'].isna().any(),
    }
    for name, frame in source_frames.items()
])
display(source_contract)
assert source_contract['contract_valid'].all()

,split_name,row_count,created_date_dtype,timezone,null_count,contract_valid
0,train,23699,"datetime64[us, UTC]",UTC,0,True
1,validation,6762,"datetime64[us, UTC]",UTC,0,True
2,test,5499,"datetime64[us, UTC]",UTC,0,True


### D. Temporal derivation rules

All definitions are pure calendar extraction from `created_date`: hour uses 0–23; weekday uses Pandas' Monday=0 through Sunday=6 convention; month uses 1–12; weekend is true exactly for weekday values 5 and 6.

In [13]:
derivation_rules = pd.DataFrame([
    {'feature_name': 'created_hour', 'source': 'created_date', 'rule': 'UTC hour component', 'dtype': 'Int8', 'domain': '0..23'},
    {'feature_name': 'created_day_of_week', 'source': 'created_date', 'rule': 'Monday=0 through Sunday=6', 'dtype': 'Int8', 'domain': '0..6'},
    {'feature_name': 'created_month', 'source': 'created_date', 'rule': 'calendar month number', 'dtype': 'Int8', 'domain': '1..12'},
    {'feature_name': 'is_weekend', 'source': 'created_date', 'rule': 'created_day_of_week in {5, 6}', 'dtype': 'bool', 'domain': '{False, True}'},
])
display(derivation_rules)

,feature_name,source,rule,dtype,domain
0,created_hour,created_date,UTC hour component,Int8,0..23
1,created_day_of_week,created_date,Monday=0 through Sunday=6,Int8,0..6
2,created_month,created_date,calendar month number,Int8,1..12
3,is_weekend,created_date,"created_day_of_week in {5, 6}",bool,"{False, True}"


### E. Training derivation preview

This small preview is calculated from actual training timestamps. It is evidence of the reusable builder's output, not hardcoded example data.

In [14]:
derived_frames = derive_split_temporal_features(source_frames, policy=policy)
preview_columns = ['created_date', *approved_names]
display(derived_frames['train'][preview_columns].head(8))

,created_date,created_hour,created_day_of_week,created_month,is_weekend
0,2024-01-01 07:58:15+00:00,7,0,1,False
1,2024-01-01 08:00:34+00:00,8,0,1,False
2,2024-01-01 08:01:53+00:00,8,0,1,False
3,2024-01-01 08:03:15+00:00,8,0,1,False
4,2024-01-01 08:03:59+00:00,8,0,1,False
5,2024-01-01 08:04:35+00:00,8,0,1,False
6,2024-01-01 08:05:18+00:00,8,0,1,False
7,2024-01-01 08:06:21+00:00,8,0,1,False


### F. Split consistency and feature-domain validation

The identical builder is applied independently to train, validation, and test. Later splits validate domains only; they do not select features or alter derivation rules. Repeatability compares values exactly across two independent calls.

In [15]:
feature_validation = build_temporal_validation_table(
    source_frames, derived_frames, policy=policy
)
display(feature_validation)
assert feature_validation['creation_status'].eq('COMPLETE').all()
assert feature_validation['domain_valid'].all()
assert feature_validation['deterministic'].all()
weekend_check = feature_validation.loc[
    feature_validation['feature_name'].eq('is_weekend'),
    'weekend_relationship_valid',
]
assert weekend_check.eq(True).all()

,feature_name,source_column,policy_status,created,dtype,non_null_count,unique_count,minimum,maximum,domain_valid,train_valid,validation_valid,test_valid,weekend_relationship_valid,deterministic,creation_status
0,created_hour,created_date,APPROVED_CANDIDATE,True,Int8,23699,24,0,23,True,True,True,True,None,True,COMPLETE
1,created_day_of_week,created_date,APPROVED_CANDIDATE,True,Int8,23699,7,0,6,True,True,True,True,None,True,COMPLETE
2,created_month,created_date,APPROVED_CANDIDATE,True,Int8,23699,12,1,12,True,True,True,True,None,True,COMPLETE
3,is_weekend,created_date,APPROVED_CANDIDATE,True,bool,23699,2,False,True,True,True,True,True,True,True,COMPLETE


### G. Row and target reconciliation

Feature creation may add approved columns only. Row count, index, order, complaint identifier, target, and every original source value must reconcile exactly.

In [16]:
feature_reconciliation = build_feature_reconciliation_table(
    source_frames,
    derived_frames,
    identifier_column=eda_config.identifier_column,
    target_column=eda_config.target_column,
)
display(feature_reconciliation)
reconciliation_columns = [
    'row_count_preserved', 'index_preserved', 'row_order_preserved',
    'unique_key_preserved', 'target_preserved', 'source_values_preserved',
]
assert feature_reconciliation[reconciliation_columns].all().all()

,split_name,rows_before,rows_after,row_count_preserved,index_preserved,row_order_preserved,unique_key_preserved,target_preserved,source_values_preserved
0,train,23699,23699,True,True,True,True,True,True
1,validation,6762,6762,True,True,True,True,True,True
2,test,5499,5499,True,True,True,True,True,True


### H. Source immutability and implementation boundary

All derived frames exist in memory only. Source DataFrames, split artifacts, Notebook 10 reports, and the frozen policy must remain byte-for-byte and modification-time unchanged. Non-approved temporal fields are not created, and conditional geography is not promoted or transformed.

In [17]:
repeat_frames = derive_split_temporal_features(source_frames, policy=policy)
creation_after_states = {str(path.relative_to(PROJECT_ROOT)): file_state(path) for path in governed_inputs}
source_frames_unchanged = all(
    source_frames[name].equals(source_snapshots[name]) for name in source_frames
)
repeat_identical = all(
    derived_frames[name].equals(repeat_frames[name]) for name in derived_frames
)
new_columns = {
    name: tuple(column for column in derived_frames[name] if column not in source_frames[name])
    for name in source_frames
}
only_approved_created = all(columns == approved_names for columns in new_columns.values())
creation_boundary_checks = pd.Series({
    'source_frames_unchanged': source_frames_unchanged,
    'repeat_derivation_identical': repeat_identical,
    'only_approved_features_created': only_approved_created,
    'governed_files_unchanged': before_states == creation_after_states,
    'no_rows_added_or_removed': feature_reconciliation['row_count_preserved'].all(),
    'no_preprocessing_or_modelling': all(policy.implementation_boundary[flag] is False for flag in boundary_flags),
}, name='passed')
display(creation_boundary_checks)
assert creation_boundary_checks.all()

source_frames_unchanged           True
repeat_derivation_identical       True
only_approved_features_created    True
governed_files_unchanged          True
no_rows_added_or_removed          True
no_preprocessing_or_modelling     True
Name: passed, dtype: bool

### I. Completion decision

Deterministic feature creation is complete when policy authority, source contracts, exact repeatability, domains, reconciliation, and immutability all pass. The result remains an in-memory analytical view, not a persisted model matrix.

The next reviewed work, categorical missing-value handling, begins below.

In [18]:
creation_complete = (
    feature_validation['creation_status'].eq('COMPLETE').all()
    and feature_reconciliation[reconciliation_columns].all().all()
    and creation_boundary_checks.all()
)
pd.Series({
    'creation_decision': 'COMPLETE' if creation_complete else 'NOT_COMPLETE',
    'model_ready': False,
    'created_features': ', '.join(approved_names),
    'next_work': 'Handle Categorical Missing Values',
})

creation_decision                                             COMPLETE
model_ready                                                      False
created_features     created_hour, created_day_of_week, created_mon...
next_work                            Handle Categorical Missing Values
dtype: object

## Categorical Missing-Value Handling

### A. Scope boundary

The frozen eligibility policy remains authoritative. This section defines how an approved categorical feature would represent a genuinely absent source value, but it does not promote conditional geography. Upstream validation/cleaning owns malformed values, whitespace and blank normalization, coordinate validity, and audit counts; feature preprocessing handles only legitimate missing values that remain afterward. Numeric filling, rare grouping, unseen-category handling, encoding, scaling, combined preprocessing, persisted model matrices, and modelling remain outside this work.

In [19]:
from urban_ops.cleaning.pipeline import load_cleaning_config
from urban_ops.features.categorical_missing import (
    active_categorical_feature_names,
    build_categorical_missing_evidence,
    build_categorical_reconciliation_table,
    load_categorical_missing_config,
    replace_categorical_missing,
    replace_split_categorical_missing,
)

MISSING_CONFIG_PATH = PROJECT_ROOT / 'configs/features/resolution_risk_categorical_missing.yaml'
CLEANING_RULES_PATH = PROJECT_ROOT / 'configs/data/cleaning_rules.yaml'
missing_config = load_categorical_missing_config(MISSING_CONFIG_PATH)
cleaning_config = load_cleaning_config(CLEANING_RULES_PATH)
numeric_missingness_decision = policy.implementation_boundary['numeric_missingness']
categorical_governed_inputs = tuple(dict.fromkeys((*governed_inputs, MISSING_CONFIG_PATH, CLEANING_RULES_PATH, PROJECT_ROOT / 'src/urban_ops/features/temporal.py')))
categorical_before_states = {
    str(path.relative_to(PROJECT_ROOT)): file_state(path)
    for path in categorical_governed_inputs
}
active_categorical_names = active_categorical_feature_names(policy, missing_config)
pd.Series({
    'strategy': missing_config.strategy,
    'missing_token': missing_config.token,
    'supported_columns': ', '.join(missing_config.supported_columns),
    'active_columns': ', '.join(active_categorical_names) or 'NONE',
    'incident_zip_trimmed_upstream': 'incident_zip' in cleaning_config.trim_columns,
    'incident_zip_blanks_null_upstream': 'incident_zip' in cleaning_config.blank_to_null_columns,
    'numeric_missingness_status': numeric_missingness_decision['status'],
})

strategy                                                         constant
missing_token                                                 __MISSING__
supported_columns                    borough, location_type, incident_zip
active_columns                                                       NONE
incident_zip_trimmed_upstream                                        True
incident_zip_blanks_null_upstream                                    True
numeric_missingness_status                                       DEFERRED
dtype: object

### B. Why missingness remains explicit

A categorical missing value is not replaced with the most common category because that would fabricate information. `__MISSING__` means only that the source did not provide a value. It is distinct from `__RARE__`, a known low-support training category, and `__UNKNOWN__`, a real category unseen during training. Rare and unseen behavior is not implemented here.

### C. Active categorical policy status

Handler support and baseline activation are different decisions. The configuration supports the three reviewed categorical geography fields; the frozen policy alone decides whether any is active.

In [20]:
categorical_policy_table = pd.DataFrame([
    {
        'feature_name': name,
        'policy_status': policy.by_name[name].policy_status.value,
        'prediction_time_status': policy.by_name[name].prediction_time_status,
        'handler_supported': True,
        'active': name in active_categorical_names,
    }
    for name in missing_config.supported_columns
])
display(categorical_policy_table)
assert categorical_policy_table['policy_status'].eq('CONDITIONAL').all()
assert categorical_policy_table['active'].eq(False).all()

,feature_name,policy_status,prediction_time_status,handler_supported,active
0,borough,CONDITIONAL,UNRESOLVED,True,False
1,location_type,CONDITIONAL,UNRESOLVED,True,False
2,incident_zip,CONDITIONAL,UNRESOLVED,True,False


### D. Missingness before transformation

These counts come from the authoritative split views. They describe the data but cannot change eligibility or the constant rule.

In [21]:
missingness_before = pd.DataFrame([
    {
        'split_name': split_name,
        'feature_name': feature_name,
        'missing_count': int(frame[feature_name].isna().sum()),
        'missing_rate': int(frame[feature_name].isna().sum()) / len(frame),
        'dtype': str(frame[feature_name].dtype),
    }
    for split_name, frame in derived_frames.items()
    for feature_name in missing_config.supported_columns
])
display(missingness_before)

,split_name,feature_name,missing_count,missing_rate,dtype
0,train,borough,0,0.000000,string
1,train,location_type,1965,0.082915,string
2,train,incident_zip,2,0.000084,string
3,validation,borough,0,0.000000,string
4,validation,location_type,686,0.101449,string
5,validation,incident_zip,1,0.000148,string
6,test,borough,0,0.000000,string
7,test,location_type,642,0.116748,string
8,test,incident_zip,0,0.000000,string


### E. Canonical `__MISSING__` rule

For a policy-approved categorical field, the deterministic rule is `null → __MISSING__`. No mode, category probability, target rate, row order, randomness, or split-specific token is used. `incident_zip` remains categorical text; it is never averaged, interpolated, or treated as continuous.

In [22]:
controlled_example = pd.DataFrame({
    'location_type': pd.Series(['Residential', pd.NA, 'Commercial'], dtype='string'),
    'incident_zip': pd.Series(['00123', pd.NA, '10099'], dtype='string'),
})
controlled_result = replace_categorical_missing(
    controlled_example,
    columns=['location_type', 'incident_zip'],
    missing_token=missing_config.token,
)
display(controlled_result)
assert controlled_result['location_type'].tolist() == ['Residential', '__MISSING__', 'Commercial']
assert controlled_result['incident_zip'].tolist() == ['00123', '__MISSING__', '10099']

,location_type,incident_zip
0,Residential,00123
1,__MISSING__,__MISSING__
2,Commercial,10099


### F. Training demonstration

The actual training preview below selects rows with at least one reviewed categorical null. Because all reviewed fields remain conditional, the governed working view intentionally preserves those nulls rather than silently activating the fields.

In [23]:
categorical_frames = replace_split_categorical_missing(
    derived_frames, policy=policy, config=missing_config
)
reviewed_columns = list(missing_config.supported_columns)
training_missing_mask = derived_frames['train'][reviewed_columns].isna().any(axis=1)
actual_training_preview = categorical_frames['train'].loc[
    training_missing_mask, reviewed_columns
].head(8)
display(actual_training_preview)

,borough,location_type,incident_zip
19,BROOKLYN,<NA>,11218
51,MANHATTAN,<NA>,10001
77,BROOKLYN,<NA>,11218
134,BROOKLYN,<NA>,11221
164,BROOKLYN,<NA>,11211
184,MANHATTAN,<NA>,10032
199,BRONX,<NA>,10466
200,BROOKLYN,<NA>,11211


### G. Split consistency and reconciliation

The same policy-driven function is invoked for train, validation, and test. Evidence reports actual before/after counts and verifies identity, order, and target preservation.

In [24]:
categorical_evidence = build_categorical_missing_evidence(
    derived_frames, categorical_frames, policy=policy, config=missing_config
)
categorical_reconciliation = build_categorical_reconciliation_table(
    derived_frames,
    categorical_frames,
    identifier_column=eda_config.identifier_column,
    target_column=eda_config.target_column,
)
display(categorical_evidence)
display(categorical_reconciliation)
categorical_reconciliation_columns = [
    'row_count_preserved', 'index_preserved', 'row_order_preserved',
    'unique_key_preserved', 'target_preserved',
]
assert categorical_evidence['status'].eq('PASS').all()
assert categorical_reconciliation[categorical_reconciliation_columns].all().all()

,split_name,feature_name,policy_status,active,missing_count_before,missing_rate_before,source_token_collision,missing_token_count_after,null_count_after,row_count,dtype_before,dtype_after,categorical_dtype_preserved,transformation_applied,reconciled,status
0,train,borough,CONDITIONAL,False,0,0.000000,False,0,0,23699,string,string,True,False,True,PASS
1,train,location_type,CONDITIONAL,False,1965,0.082915,False,0,1965,23699,string,string,True,False,True,PASS
2,train,incident_zip,CONDITIONAL,False,2,0.000084,False,0,2,23699,string,string,True,False,True,PASS
3,validation,borough,CONDITIONAL,False,0,0.000000,False,0,0,6762,string,string,True,False,True,PASS
4,validation,location_type,CONDITIONAL,False,686,0.101449,False,0,686,6762,string,string,True,False,True,PASS
5,validation,incident_zip,CONDITIONAL,False,1,0.000148,False,0,1,6762,string,string,True,False,True,PASS
6,test,borough,CONDITIONAL,False,0,0.000000,False,0,0,5499,string,string,True,False,True,PASS
7,test,location_type,CONDITIONAL,False,642,0.116748,False,0,642,5499,string,string,True,False,True,PASS
8,test,incident_zip,CONDITIONAL,False,0,0.000000,False,0,0,5499,string,string,True,False,True,PASS


,split_name,rows_before,rows_after,row_count_preserved,index_preserved,row_order_preserved,unique_key_preserved,target_preserved
0,train,23699,23699,True,True,True,True,True
1,validation,6762,6762,True,True,True,True,True
2,test,5499,5499,True,True,True,True,True


### H. Token-collision validation

A genuine source category already equal to `__MISSING__` would be ambiguous and must fail. The authoritative split views are checked before any future activation.

In [25]:
collision_check = categorical_evidence.pivot(
    index='feature_name', columns='split_name', values='source_token_collision'
)
display(collision_check)
assert not categorical_evidence['source_token_collision'].any()

split_name,test,train,validation
feature_name,,,
borough,False,False,False
incident_zip,False,False,False
location_type,False,False,False


### I. Source immutability

Only in-memory copies are produced. Repeat application is idempotent while DataFrame `attrs` provenance remains in memory. CSV, Parquet, or another serialization/reload may lose that provenance; reapplication then fails loudly on an unmarked `__MISSING__` collision as an intentional fail-safe. The source DataFrames, split files, metadata, latest pointer, Notebook 10 reports, frozen policy, cleaning rules, and deterministic temporal implementation must remain unchanged.

In [26]:
categorical_repeat = replace_split_categorical_missing(
    categorical_frames, policy=policy, config=missing_config
)
categorical_after_states = {
    str(path.relative_to(PROJECT_ROOT)): file_state(path)
    for path in categorical_governed_inputs
}
categorical_boundary_checks = pd.Series({
    'source_frames_unchanged': all(source_frames[name].equals(source_snapshots[name]) for name in source_frames),
    'repeat_application_identical': all(categorical_frames[name].equals(categorical_repeat[name]) for name in categorical_frames),
    'conditional_features_not_activated': not active_categorical_names,
    'governed_files_unchanged': categorical_before_states == categorical_after_states,
    'no_rows_added_or_removed': categorical_reconciliation['row_count_preserved'].all(),
    'temporal_features_complete': all(
        not categorical_frames[name][feature].isna().any()
        for name in categorical_frames
        for feature in approved_names
    ),
    'latitude_unchanged': all(derived_frames[name]['latitude'].equals(categorical_frames[name]['latitude']) for name in derived_frames),
    'longitude_unchanged': all(derived_frames[name]['longitude'].equals(categorical_frames[name]['longitude']) for name in derived_frames),
}, name='passed')
display(categorical_boundary_checks)
assert categorical_boundary_checks.all()

source_frames_unchanged               True
repeat_application_identical          True
conditional_features_not_activated    True
governed_files_unchanged              True
no_rows_added_or_removed              True
temporal_features_complete            True
latitude_unchanged                    True
longitude_unchanged                   True
Name: passed, dtype: bool

### J. Deferred numeric missingness

The governed policy records `latitude` and `longitude` as feature status `CONDITIONAL`, prediction-time status `UNRESOLVED`, and missing-value handling status `DEFERRED`. No coordinate imputation or missingness indicator is implemented. Revisit numeric handling only if a coordinate becomes `APPROVED_CANDIDATE`. The future design must validate the coordinate pair upstream, normalize invalid or impossible coordinates under validation/cleaning policy with audit counts, fit imputation values on training data only, apply those fitted values to validation/test/inference, evaluate `latitude_missing` and `longitude_missing`, avoid implying a placeholder is the complaint's true location, and persist fitted preprocessing with the model pipeline.

### K. Completion decision

Completion requires a validated constant token, a working generic rule, truthful policy-controlled inactivity, collision-free source evidence, exact row reconciliation, repeatability, and source immutability.

**STOP:** the next reviewed work is **Handle Rare and Unseen Categories**. It is intentionally not implemented here.

In [27]:
categorical_complete = (
    missing_config.strategy == 'constant'
    and missing_config.token == '__MISSING__'
    and controlled_result.isna().sum().sum() == 0
    and categorical_evidence['status'].eq('PASS').all()
    and not categorical_evidence['source_token_collision'].any()
    and categorical_reconciliation[categorical_reconciliation_columns].all().all()
    and categorical_boundary_checks.all()
    and 'incident_zip' in cleaning_config.trim_columns
    and 'incident_zip' in cleaning_config.blank_to_null_columns
    and numeric_missingness_decision['status'] == 'DEFERRED'
    and numeric_missingness_decision['trigger_feature_status'] == 'APPROVED_CANDIDATE'
)
pd.Series({
    'categorical_integrity': 'PASS' if categorical_evidence['status'].eq('PASS').all() else 'FAIL',
    'categorical_missing_policy': 'PASS' if missing_config.token == '__MISSING__' else 'FAIL',
    'row_reconciliation': 'PASS' if categorical_reconciliation[categorical_reconciliation_columns].all().all() else 'FAIL',
    'source_immutability': 'PASS' if categorical_boundary_checks['governed_files_unchanged'] else 'FAIL',
    'incident_zip_blank_normalization': 'PASS' if 'incident_zip' in cleaning_config.blank_to_null_columns else 'FAIL',
    'numeric_missingness_status': numeric_missingness_decision['status'],
    'numeric_imputation_implemented': False,
    'categorical_missing_decision': 'COMPLETE' if categorical_complete else 'NOT_COMPLETE',
    'next_work': 'Handle Rare and Unseen Categories',
})

categorical_integrity                                            PASS
categorical_missing_policy                                       PASS
row_reconciliation                                               PASS
source_immutability                                              PASS
incident_zip_blank_normalization                                 PASS
numeric_missingness_status                                   DEFERRED
numeric_imputation_implemented                                  False
categorical_missing_decision                                 COMPLETE
next_work                           Handle Rare and Unseen Categories
dtype: object

## Rare and Unseen Categorical Handling

### A. Scope and production decision

Phase 4 introduces an explicit training-only fit/transform lifecycle without changing the frozen feature policy. `borough`, `location_type`, and `incident_zip` remain `CONDITIONAL`, so the real baseline has no active categorical feature and the production result is **COMPLETE — GOVERNED NO-OP**. Handler support never implies activation. Categorical encoding, combined preprocessing, persistence, and modelling remain outside this phase.

In [28]:
from urban_ops.features.rare_unseen import (
    active_rare_unseen_feature_names,
    build_rare_unseen_evidence,
    fit_rare_unseen_handler,
    load_rare_unseen_config,
    transform_split_rare_unseen,
)

CARDINALITY_CONFIG_PATH = PROJECT_ROOT / 'configs/features/resolution_risk_categorical_cardinality.yaml'
cardinality_config = load_rare_unseen_config(
    CARDINALITY_CONFIG_PATH, missing_config=missing_config
)
phase_4_governed_inputs = tuple(dict.fromkeys((
    *categorical_governed_inputs,
    CARDINALITY_CONFIG_PATH,
    PROJECT_ROOT / 'src/urban_ops/features/rare_unseen.py',
)))
phase_4_before_states = {
    str(path.relative_to(PROJECT_ROOT)): file_state(path)
    for path in phase_4_governed_inputs
}
active_cardinality_names = active_rare_unseen_feature_names(
    policy, cardinality_config
)
display(pd.Series({
    'strategy': cardinality_config.strategy,
    'candidate_min_count': cardinality_config.min_count,
    'missing_token': cardinality_config.missing_token,
    'rare_token': cardinality_config.rare_token,
    'unknown_token': cardinality_config.unknown_token,
    'active_categorical_features': ', '.join(active_cardinality_names) or 'NONE',
    'production_decision': cardinality_config.production_decision,
}))

strategy                        minimum_count
candidate_min_count                        10
missing_token                     __MISSING__
rare_token                           __RARE__
unknown_token                     __UNKNOWN__
active_categorical_features              NONE
production_decision            GOVERNED_NO_OP
dtype: object

### B. Training-only cardinality and threshold evidence

Notebook 10 evaluated absolute counts 10, 25, and 50 and relative frequencies 0.1%, 0.5%, and 1% from training only. `borough` and `location_type` are low-cardinality and group no categories at count 10. Conditional `incident_zip` has 175 training categories; count 10 would group 33 categories covering 120 rows (0.506%). The lowest evaluated absolute threshold is retained as a simple candidate, but it is not an active production transformation and must be reviewed if policy later approves a categorical field.

In [29]:
rare_threshold_table = pd.read_csv(PROJECT_ROOT / cardinality_config.evidence_source)
selected_threshold_evidence = rare_threshold_table.loc[
    rare_threshold_table['feature_name'].isin(cardinality_config.supported_columns)
    & rare_threshold_table['threshold_type'].eq('count')
    & rare_threshold_table['threshold_value'].eq(cardinality_config.min_count),
    [
        'feature_name', 'threshold_type', 'threshold_value',
        'rare_category_count', 'train_rows_affected', 'train_row_share',
    ],
].reset_index(drop=True)
display(selected_threshold_evidence)
assert set(cardinality_config.evaluated_count_thresholds) == {10, 25, 50}
assert set(cardinality_config.evaluated_frequency_thresholds) == {0.001, 0.005, 0.01}
assert selected_threshold_evidence['feature_name'].nunique() == 3

,feature_name,threshold_type,threshold_value,rare_category_count,train_rows_affected,train_row_share
0,borough,count,10.0,0,0,0.000000
1,location_type,count,10.0,0,0,0.000000
2,incident_zip,count,10.0,33,120,0.005064


### C. Separate missing, rare, and unseen meanings

Phase 4 preserves the Phase 3 token exactly: `__MISSING__` remains source absence. A genuine training category below the frozen count threshold maps to `__RARE__`. A genuine validation, test, or inference category absent from the complete training vocabulary maps to `__UNKNOWN__`. Unseen categories are never treated as missing or rare, and raw rare/unknown token collisions fail safely.

### D. Explicit training-only fitted state

Only the training frame enters `fit_rare_unseen_handler`. The frozen result contains deterministic category counts, the complete training vocabulary, retained and rare categories, tokens, threshold, and policy/config versions. Validation and test enter only the transform function, which cannot learn or update state.

In [30]:
fitted_cardinality = fit_rare_unseen_handler(
    categorical_frames['train'], policy=policy, config=cardinality_config
)
fitted_state_before_transform = fitted_cardinality.to_dict()
fitted_fingerprint_before_transform = fitted_cardinality.fingerprint
phase_4_frames = transform_split_rare_unseen(
    categorical_frames, fitted=fitted_cardinality, config=cardinality_config
)
rare_unseen_evidence = build_rare_unseen_evidence(
    categorical_frames,
    phase_4_frames,
    fitted=fitted_cardinality,
    policy=policy,
    config=cardinality_config,
)
rare_unseen_reconciliation = build_categorical_reconciliation_table(
    categorical_frames,
    phase_4_frames,
    identifier_column=eda_config.identifier_column,
    target_column=eda_config.target_column,
)
display(rare_unseen_evidence)
display(rare_unseen_reconciliation)

,feature_name,policy_status,active,training_row_count,training_distinct_count,retained_category_count,rare_category_count,train_rows_mapped_rare,train_rare_fraction,validation_unseen_value_count,validation_rows_mapped_unknown,test_unseen_value_count,test_rows_mapped_unknown,transformation_applied,reconciled,status
0,borough,CONDITIONAL,False,23699,5,0,0,0,0.0,0,0,0,0,False,True,PASS
1,location_type,CONDITIONAL,False,23699,3,0,0,0,0.0,0,0,0,0,False,True,PASS
2,incident_zip,CONDITIONAL,False,23699,174,0,0,0,0.0,3,0,5,0,False,True,PASS


,split_name,rows_before,rows_after,row_count_preserved,index_preserved,row_order_preserved,unique_key_preserved,target_preserved
0,train,23699,23699,True,True,True,True,True
1,validation,6762,6762,True,True,True,True,True
2,test,5499,5499,True,True,True,True,True


### E. Governed no-op evidence and reconciliation

The evidence reports actual training cardinality and descriptive later unseen categories, while correctly recording zero mapped rows for inactive features. No conditional feature is promoted. Row count, index, order, identifier, target, unrelated columns, split artifacts, and fitted state must remain unchanged. Synthetic-policy unit and integration tests separately execute the active rare/unknown branches.

In [31]:
repeated_fitted_cardinality = fit_rare_unseen_handler(
    categorical_frames['train'], policy=policy, config=cardinality_config
)
repeated_phase_4_frames = transform_split_rare_unseen(
    phase_4_frames, fitted=fitted_cardinality, config=cardinality_config
)
phase_4_after_states = {
    str(path.relative_to(PROJECT_ROOT)): file_state(path)
    for path in phase_4_governed_inputs
}
rare_unseen_reconciliation_columns = [
    'row_count_preserved', 'index_preserved', 'row_order_preserved',
    'unique_key_preserved', 'target_preserved',
]
phase_4_checks = pd.Series({
    'production_is_governed_no_op': cardinality_config.production_decision == 'GOVERNED_NO_OP',
    'no_active_categorical_features': not active_cardinality_names,
    'no_fitted_column_state': not fitted_cardinality.columns,
    'deterministic_fit': fitted_cardinality == repeated_fitted_cardinality,
    'fitted_state_immutable': (
        fitted_cardinality.to_dict() == fitted_state_before_transform
        and fitted_cardinality.fingerprint == fitted_fingerprint_before_transform
    ),
    'repeat_transform_identical': all(
        phase_4_frames[name].equals(repeated_phase_4_frames[name])
        for name in phase_4_frames
    ),
    'conditional_values_unchanged': all(
        categorical_frames[split][feature].equals(phase_4_frames[split][feature])
        for split in phase_4_frames
        for feature in cardinality_config.supported_columns
    ),
    'evidence_reconciled': rare_unseen_evidence['status'].eq('PASS').all(),
    'identity_and_target_preserved': rare_unseen_reconciliation[rare_unseen_reconciliation_columns].all().all(),
    'governed_files_unchanged': phase_4_before_states == phase_4_after_states,
}, name='passed')
display(phase_4_checks)
assert phase_4_checks.all()

production_is_governed_no_op      True
no_active_categorical_features    True
no_fitted_column_state            True
deterministic_fit                 True
fitted_state_immutable            True
repeat_transform_identical        True
conditional_values_unchanged      True
evidence_reconciled               True
identity_and_target_preserved     True
governed_files_unchanged          True
Name: passed, dtype: bool

### F. Phase 4 completion decision

Phase 4 is complete when token semantics are distinct, fitted state is explicit and training-only, transforms cannot update it, evidence reconciles, source artifacts remain immutable, and the real policy activates no conditional field.

**STOP:** no `OneHotEncoder`, `OrdinalEncoder`, `ColumnTransformer`, model pipeline, scaling, or training is implemented.

Next work: **Categorical encoding**, implemented below.

In [32]:
phase_4_complete = (
    len({
        cardinality_config.missing_token,
        cardinality_config.rare_token,
        cardinality_config.unknown_token,
    }) == 3
    and rare_unseen_evidence['status'].eq('PASS').all()
    and rare_unseen_reconciliation[rare_unseen_reconciliation_columns].all().all()
    and phase_4_checks.all()
)
pd.Series({
    'phase': 'Phase 4 — Rare and Unseen Categorical Handling',
    'active_categorical_features': ', '.join(active_cardinality_names) or 'NONE',
    'fit_authority': 'TRAIN ONLY',
    'conditional_features_activated': False,
    'phase_3_missing_semantics_preserved': True,
    'latitude_longitude_missingness': 'DEFERRED',
    'categorical_encoding_implemented': False,
    'decision': 'COMPLETE — GOVERNED NO-OP' if phase_4_complete else 'NOT COMPLETE',
    'next_work': 'Categorical encoding',
})

phase                                  Phase 4 — Rare and Unseen Categorical Handling
active_categorical_features                                                      NONE
fit_authority                                                              TRAIN ONLY
conditional_features_activated                                                  False
phase_3_missing_semantics_preserved                                              True
latitude_longitude_missingness                                               DEFERRED
categorical_encoding_implemented                                                False
decision                                                    COMPLETE — GOVERNED NO-OP
next_work                                                        Categorical encoding
dtype: object

## Categorical Encoding

Phase 5 converts approved categorical Phase 4 outputs into a sparse numeric representation for later baseline models. The encoder is fitted from training-derived state only and does not activate conditional features, build a combined preprocessing matrix, process numeric fields, or train a model.


### A. Scope and production decision

The real production policy is still authoritative. `borough`, `location_type`, and `incident_zip` are supported by the encoder, but support is not approval. Because all supported categorical fields remain conditional, production encoding is expected to be a governed no-op. Synthetic-policy tests separately execute the active one-hot path.


In [33]:
from urban_ops.features.categorical_encoding import (
    active_encoding_feature_names,
    build_categorical_encoding_evidence,
    fit_categorical_encoder,
    load_categorical_encoding_config,
    transform_split_categorical_encoder,
)

ENCODING_CONFIG_PATH = PROJECT_ROOT / 'configs/features/resolution_risk_categorical_encoding.yaml'
encoding_config = load_categorical_encoding_config(
    ENCODING_CONFIG_PATH, cardinality_config=cardinality_config
)
active_encoding_names = active_encoding_feature_names(policy, encoding_config)
phase_5_governed_inputs = tuple(dict.fromkeys((
    *phase_4_governed_inputs,
    ENCODING_CONFIG_PATH,
    PROJECT_ROOT / 'src/urban_ops/features/categorical_encoding.py',
)))
phase_5_before_states = {
    str(path.relative_to(PROJECT_ROOT)): file_state(path)
    for path in phase_5_governed_inputs
}

pd.Series({
    'strategy': encoding_config.strategy,
    'handle_unknown': encoding_config.handle_unknown,
    'drop': encoding_config.drop,
    'sparse_output': encoding_config.sparse_output,
    'reserved_token_policy': 'explicit __MISSING__, __RARE__, __UNKNOWN__ columns',
    'production_decision': encoding_config.production_decision,
    'active_categorical_features': ', '.join(active_encoding_names) or 'NONE',
})


strategy                                                                 one_hot
handle_unknown                                                            ignore
drop                                                                        None
sparse_output                                                               True
reserved_token_policy          explicit __MISSING__, __RARE__, __UNKNOWN__ co...
production_decision                                               GOVERNED_NO_OP
active_categorical_features                                                 NONE
dtype: object

### B. One-hot strategy and token vocabulary

The governed encoder uses sklearn `OneHotEncoder` with `handle_unknown="ignore"`, `drop=None`, and sparse output. Phase 4 remains responsible for mapping legitimate later unseen values to `__UNKNOWN__`; `handle_unknown="ignore"` is only a defensive fallback if an unexpected raw value reaches the encoder.

For each active feature, the encoded vocabulary is built from retained training categories plus explicit `__MISSING__`, `__RARE__`, and `__UNKNOWN__` columns. These sentinel columns are included even when a token is absent from the current training rows, so later validation/test/inference rows keep a stable semantic schema.


### C. Training-only fitted encoder state

`fit_categorical_encoder` receives only the training frame after Phase 4 and the immutable Phase 4 fitted state. The fitted encoder records policy/config versions, Phase 4 fingerprint, active columns, categories, feature names, sparse behavior, and a deterministic fingerprint. Validation and test use only transform.


In [34]:
fitted_encoder = fit_categorical_encoder(
    phase_4_frames['train'],
    policy=policy,
    config=encoding_config,
    fitted_cardinality=fitted_cardinality,
)
fitted_encoder_before_transform = fitted_encoder.to_dict()
fitted_encoder_fingerprint_before_transform = fitted_encoder.fingerprint
encoded_categorical_matrices = transform_split_categorical_encoder(
    phase_4_frames, fitted=fitted_encoder, config=encoding_config
)
categorical_encoding_evidence = build_categorical_encoding_evidence(
    phase_4_frames,
    encoded_categorical_matrices,
    fitted=fitted_encoder,
    policy=policy,
    config=encoding_config,
)
encoding_matrix_summary = pd.DataFrame([
    {
        'split_name': split_name,
        'row_count': matrix.shape[0],
        'encoded_column_count': matrix.shape[1],
        'nonzero_count': matrix.nnz,
        'matrix_type': type(matrix).__name__,
    }
    for split_name, matrix in encoded_categorical_matrices.items()
])
display(categorical_encoding_evidence)
display(encoding_matrix_summary)


,feature_name,policy_status,active,source_category_count,encoder_category_count,encoded_output_column_count,reserved_token_columns_present,train_row_count,validation_row_count,test_row_count,...,validation_encoded_shape,test_encoded_shape,feature_schema_identical,train_defensive_unknown_fallback_count,validation_defensive_unknown_fallback_count,test_defensive_unknown_fallback_count,sparse_output,transformation_applied,reconciled,status
0,borough,CONDITIONAL,False,0,0,0,False,23699,6762,5499,...,"(6762, 0)","(5499, 0)",True,0,0,0,True,False,True,PASS
1,location_type,CONDITIONAL,False,0,0,0,False,23699,6762,5499,...,"(6762, 0)","(5499, 0)",True,0,0,0,True,False,True,PASS
2,incident_zip,CONDITIONAL,False,0,0,0,False,23699,6762,5499,...,"(6762, 0)","(5499, 0)",True,0,0,0,True,False,True,PASS


,split_name,row_count,encoded_column_count,nonzero_count,matrix_type
0,train,23699,0,0,csr_matrix
1,validation,6762,0,0,csr_matrix
2,test,5499,0,0,csr_matrix


### D. Schema, sparsity, and reconciliation evidence

The evidence reports active/inactive status, category counts, encoded width, sentinel-column presence, sparse matrix shapes, schema consistency, and defensive unknown fallback counts. In production today, every supported categorical feature is inactive and the encoded categorical block has zero columns.


In [35]:
repeated_fitted_encoder = fit_categorical_encoder(
    phase_4_frames['train'],
    policy=policy,
    config=encoding_config,
    fitted_cardinality=fitted_cardinality,
)
repeated_encoded_categorical_matrices = transform_split_categorical_encoder(
    phase_4_frames, fitted=fitted_encoder, config=encoding_config
)
phase_5_after_states = {
    str(path.relative_to(PROJECT_ROOT)): file_state(path)
    for path in phase_5_governed_inputs
}
matrix_repetition_equal = all(
    (encoded_categorical_matrices[split] != repeated_encoded_categorical_matrices[split]).nnz == 0
    for split in encoded_categorical_matrices
)
phase_5_checks = pd.Series({
    'production_is_governed_no_op': not active_encoding_names,
    'no_encoded_production_columns': fitted_encoder.encoded_feature_count == 0,
    'fit_authority_is_train_only': fitted_encoder.cardinality_fingerprint == fitted_cardinality.fingerprint,
    'fitted_state_unchanged_by_transform': (
        fitted_encoder.to_dict() == fitted_encoder_before_transform
        and fitted_encoder.fingerprint == fitted_encoder_fingerprint_before_transform
    ),
    'deterministic_fit': repeated_fitted_encoder.to_dict() == fitted_encoder.to_dict(),
    'deterministic_transform': matrix_repetition_equal,
    'schemas_identical': categorical_encoding_evidence['feature_schema_identical'].all(),
    'sparse_output': all(
        type(matrix).__name__ == 'csr_matrix'
        for matrix in encoded_categorical_matrices.values()
    ),
    'row_counts_preserved': all(
        encoded_categorical_matrices[split].shape[0] == len(phase_4_frames[split])
        for split in encoded_categorical_matrices
    ),
    'evidence_reconciled': categorical_encoding_evidence['status'].eq('PASS').all(),
    'source_artifacts_unchanged': phase_5_before_states == phase_5_after_states,
    'conditional_features_activated': False,
    'numeric_preprocessing_implemented': False,
    'model_training_implemented': False,
})
display(phase_5_checks)
assert phase_5_checks.drop(labels=['conditional_features_activated', 'numeric_preprocessing_implemented', 'model_training_implemented']).all()
assert not phase_5_checks['conditional_features_activated']
assert not phase_5_checks['numeric_preprocessing_implemented']
assert not phase_5_checks['model_training_implemented']


production_is_governed_no_op            True
no_encoded_production_columns           True
fit_authority_is_train_only             True
fitted_state_unchanged_by_transform     True
deterministic_fit                       True
deterministic_transform                 True
schemas_identical                       True
sparse_output                           True
row_counts_preserved                    True
evidence_reconciled                     True
source_artifacts_unchanged              True
conditional_features_activated         False
numeric_preprocessing_implemented      False
model_training_implemented             False
dtype: bool

### E. Phase 5 completion decision

Phase 5 is complete when the encoder state is explicit, fitted from training-derived Phase 4 state only, sparse, schema-stable across splits, non-mutating, target-independent by design, and governed by the frozen production policy. The real production result is a governed no-op because there are no active categorical features.

**STOP:** no numeric preprocessing, scaling, `ColumnTransformer`, combined model matrix, logistic regression, model evaluation, or model training is implemented.

Next work: **Numeric preprocessing**.


In [36]:
phase_5_complete = (
    phase_5_checks.drop(labels=[
        'conditional_features_activated',
        'numeric_preprocessing_implemented',
        'model_training_implemented',
    ]).all()
    and not phase_5_checks['conditional_features_activated']
    and not phase_5_checks['numeric_preprocessing_implemented']
    and not phase_5_checks['model_training_implemented']
)
pd.Series({
    'phase': 'Phase 5 — Categorical Encoding',
    'active_categorical_features': ', '.join(active_encoding_names) or 'NONE',
    'strategy': encoding_config.strategy,
    'fit_authority': 'TRAIN ONLY',
    'encoded_feature_count': fitted_encoder.encoded_feature_count,
    'sparse_output': encoding_config.sparse_output,
    'feature_schema_identical': categorical_encoding_evidence['feature_schema_identical'].all(),
    'phase_3_missing_semantics_preserved': True,
    'phase_4_rare_unseen_semantics_preserved': True,
    'conditional_features_activated': False,
    'latitude_longitude_missingness': 'DEFERRED',
    'numeric_preprocessing_implemented': False,
    'model_training_implemented': False,
    'decision': 'COMPLETE -- GOVERNED NO-OP' if phase_5_complete else 'INCOMPLETE',
    'next_work': 'Numeric preprocessing',
})


phase                                      Phase 5 — Categorical Encoding
active_categorical_features                                          NONE
strategy                                                          one_hot
fit_authority                                                  TRAIN ONLY
encoded_feature_count                                                   0
sparse_output                                                        True
feature_schema_identical                                             True
phase_3_missing_semantics_preserved                                  True
phase_4_rare_unseen_semantics_preserved                              True
conditional_features_activated                                      False
latitude_longitude_missingness                                   DEFERRED
numeric_preprocessing_implemented                                   False
model_training_implemented                                          False
decision                              